In [2]:
import pandas as pd
import polars as pl
import numpy as np
import re 
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

In [39]:
import sys
sys.path.append("/home/a379i/Scripts")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *

/home/a379i/Scripts/utils/load_gtf_cgc_dresden.py:106: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Extended predisp' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  cgc.loc[cgc["geneID_short"].isin(extended_dresden_dt["geneID_short"]), "Predisposition"] = "Extended predisp"


In [19]:

sa = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv", sep="\t")

pr_res_path = "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/py_outrider_runs/all_cohorts/oht_cov_diag_lr_0_0001_epoc200_gpu/or_variants.parquet"
needed_cols = ["sampleID", "zScore", "proteinID", "PROTEIN_INT", "PROTEIN_LOG2INT"]

pr_output_name = "cov_gaussian_gs_lr_0_001_epoc2000_noInitPCA"

pr_res = pl.scan_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/protrider_runs/output_" + pr_output_name + "/protrider_summary.csv").select(needed_cols).collect().to_pandas()



In [30]:
matrix = pr_res.pivot(index='proteinID', columns='sampleID', values='PROTEIN_LOG2INT').fillna(0)


In [31]:
zscores_sf_normalized = matrix.apply(lambda x: (x - x.mean()) / x.std(), axis=1)

# 4. Apply Size Factor Normalization
# matrix.divide(series, axis=1) aligns the SampleIDs automatically
# zscores_sf_normalized = zscores.divide(sf_series, axis=1)

# 5. Convert back to Long Format
final_df = zscores_sf_normalized.reset_index().melt(
    id_vars='proteinID', 
    var_name='sampleID', 
    value_name='zScore'
)


In [40]:
gene_annot_dt["

,seqnames,start,end,strand,gene_id,gene_name,gene_type,gene_status,gene_name_orig,geneID_short
0,1,11869,14412,+,ENSG00000223972.4,DDX11L1,pseudogene,KNOWN,DDX11L1,ENSG00000223972
1,1,14363,29806,-,ENSG00000227232.4,WASH7P,pseudogene,KNOWN,WASH7P,ENSG00000227232
2,1,29554,31109,+,ENSG00000243485.2,MIR1302-11,lincRNA,NOVEL,MIR1302-11,ENSG00000243485
3,1,34554,36081,-,ENSG00000237613.2,FAM138A,lincRNA,KNOWN,FAM138A,ENSG00000237613
4,1,52473,54936,+,ENSG00000268020.2,OR4G4P,pseudogene,KNOWN,OR4G4P,ENSG00000268020
...,...,...,...,...,...,...,...,...,...,...
57815,MT,14149,14673,-,ENSG00000198695.2,MT-ND6,protein_coding,KNOWN,MT-ND6,ENSG00000198695
57816,MT,14674,14742,-,ENSG00000210194.1,MT-TE,Mt_tRNA,KNOWN,MT-TE,ENSG00000210194
57817,MT,14747,15887,+,ENSG00000198727.2,MT-CYB,protein_coding,KNOWN,MT-CYB,ENSG00000198727
57818,MT,15888,15953,+,ENSG00000210195.2,MT-TT,Mt_tRNA,KNOWN,MT-TT,ENSG00000210195


In [51]:
long_df = final_df.merge(gene_annot_dt, left_on='proteinID', how='left', right_on="gene_name")

# 4. Calculate P-values
# We use the absolute Z-score to get a two-tailed p-value
long_df['pValue'] = 2 * norm.cdf(-long_df['zScore'].abs())

# 5. Calculate Adjusted P-values (Multiple Testing Correction)
# 'fdr_by' is the Benjamini-Yekutieli method from your R snippet
_, padj, _, _ = multipletests(long_df['pValue'], method='fdr_by')
long_df['padjust'] = padj

# 6. Final Sort by Significance
long_df = long_df.sort_values('zScore')

In [53]:
long_df.to_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/protrider_runs/output_sf_zScores" + "/protrider_summary.csv", index=None)